# 01 — Explore one LUNA16 scan

Goal: load **one** CT, convert a nodule from millimetres to voxels, and draw it on the slice.
If this overlay is wrong, do not train anything yet.

Expected files:

- `data/raw/annotations.csv`
- `data/raw/subset0/*.mhd` and matching `*.raw`

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data.io import load_mhd, world_to_voxel, window_hu

raw = ROOT / "data" / "raw"
subset = raw / "subset0"
ann = pd.read_csv(raw / "annotations.csv")
mhd_files = sorted(subset.glob("*.mhd"))
print("scans found:", len(mhd_files))
print(ann.head())

In [ ]:
if not mhd_files:
    raise FileNotFoundError(f"Unzip subset0 into {subset}")

scan_path = mhd_files[0]
uid = scan_path.stem
volume, image = load_mhd(scan_path)
print("uid:", uid)
print("volume shape (z, y, x):", volume.shape)
print("spacing (x, y, z mm):", image.GetSpacing())
print("origin:", image.GetOrigin())

nodules = ann[ann["seriesuid"] == uid]
print("nodules in this scan:", len(nodules))
nodules

In [ ]:
if len(nodules) == 0:
    # Fall back to any annotated scan in subset0
    uids = {p.stem for p in mhd_files}
    nodules = ann[ann["seriesuid"].isin(uids)].head(1)
    uid = nodules.iloc[0]["seriesuid"]
    scan_path = subset / f"{uid}.mhd"
    volume, image = load_mhd(scan_path)
    print("switched to annotated scan", uid)

row = nodules.iloc[0]
xyz = (row["coordX"], row["coordY"], row["coordZ"])
vx, vy, vz = world_to_voxel(image, xyz)
print("world mm:", xyz)
print("voxel x,y,z:", vx, vy, vz, "diameter_mm:", row["diameter_mm"])

sl = window_hu(volume[vz])
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(sl, cmap="gray")
ax.scatter([vx], [vy], s=80, facecolors="none", edgecolors="lime", linewidths=2)
ax.set_title(f"{uid[:20]}…  z={vz}")
ax.axis("off")
plt.show()

The lime circle should sit on a roundish bright/soft lesion, not empty lung or bone.
If it does not, stop and fix `world_to_voxel` before training.

Next: `python -m src.train --config configs/default.yaml`